# TX–RY knob-space minima analysis (81 × 81 scan)

This notebook imports the two acquisition files produced by
`tx_ry_scan_observables.ipynb`:

- `tx_ry_scan_81x81_z0.csv` — image at the DVF rest position (`z = 0`);
- `tx_ry_scan_81x81_zfocus.csv` — image retraced to `z = -142 mm`.

Each file contains a complete `81 × 81` scan of the mirror knobs:

- `TX` from `-1.0` to `+1.0` mm;
- `RY` from `-0.040` to `+0.040` mrad.

For every point the acquisition stores the centroid, beam sizes and intensity
with their fit uncertainties, plus the foci `q_x`, `q_y` and the astigmatism
`q_y - q_x` obtained from the caustic scan.

The observables are plotted following the layout of
`notebooks/optimization/CAX_optimization.ipynb`, but the red highlight now
marks the region that is **within the measurement uncertainty of the optimum**
instead of a fixed percentage of the observable range:

- for `sigma_x`, `sigma_y`, `Intensity` and `|astigmatism|`, the band is
  `value <= min(value) + err(min)`;
- for the centroids the target is zero, so the band is `|centroid| <= err`;
- for the objectives the observable uncertainties are propagated to the
  objective (delta method), then the same `min + err(min)` rule is applied.

The objective functions `f1 … f8` keep the exact definitions and normalisation
constants used in `CAX_optimization.ipynb`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.patches import Patch

from apsuite.optimization.rcds import RCDSParams, RCDS

from shadow_interface.utils import save_image, image_from_beam
from shadow_interface.si_beamlines import CAXSim

import os
import sys

sys.path.append(os.path.expanduser("~/repos/cax-control"))

from caustic import ana_from_image, fit_beam_image

In [ ]:
DATA_DIR = Path.cwd()
if not (DATA_DIR / "tx_ry_scan_81x81_z0_20261105_203738.csv").exists():
    DATA_DIR = Path("notebooks") / "knobspace_exploration"

Z0_PATH = DATA_DIR / "tx_ry_scan_81x81_z0_20261105_203738.csv"
ZF_PATH = DATA_DIR / "tx_ry_scan_81x81_zfocus_20261105_203738.csv"

N = 81
Z_FOCUS = -142.0  # mm

# Normalisation constants kept from CAX_optimization.ipynb
I0 = 10.0
DELTA = 500.0
SIGMAX_0 = 19.9  # um
SIGMAY_0 = 11.5  # um

print(Z0_PATH)
print(ZF_PATH)

## Load the scans and reshape them into 2D arrays

`point_index` runs as `ix * 81 + iy`, with `ix` indexing `TX` and `iy`
indexing `RY`, so a plain `(81, 81)` reshape gives arrays ordered `[TX, RY]`.

In [ ]:
df_z0 = pd.read_csv(Z0_PATH).sort_values("point_index").reset_index(drop=True)
df_zf = pd.read_csv(ZF_PATH).sort_values("point_index").reset_index(drop=True)

assert len(df_z0) == N * N and len(df_zf) == N * N


def to_grid(frame, column):
    """Return a column as an (N, N) array ordered [TX, RY]."""
    return frame[column].to_numpy(dtype=float).reshape(N, N)


TX = to_grid(df_z0, "TX_mm")
RY = to_grid(df_z0, "RY_mrad")

assert np.allclose(TX, to_grid(df_zf, "TX_mm"))
assert np.allclose(RY, to_grid(df_zf, "RY_mrad"))

datasets = {
    "z = 0 mm": df_z0,
    f"z = {Z_FOCUS:g} mm": df_zf,
}

print(f"TX: {TX[0, 0]:+.3f} ... {TX[-1, 0]:+.3f} mm")
print(f"RY: {RY[0, 0]:+.4f} ... {RY[0, -1]:+.4f} mrad")
print(f"grid shape: {TX.shape}")

In [ ]:
np.sum(np.array(df_zf['duration_s']))/3600

## Uncertainty-aware highlight helpers

In [ ]:
def _legend_proxy(label):
    return Patch(facecolor="red", alpha=0.3, label=label)


def add_minimum_error_band(ax, x, y, values, errors, *, legend=True):
    """Highlight values within the measurement error of the global minimum."""
    values = np.asarray(values, dtype=float)
    errors = np.asarray(errors, dtype=float)

    finite = np.where(np.isfinite(values), values, np.inf)
    idx = np.unravel_index(np.argmin(finite), values.shape)
    minimum = values[idx]
    error = errors[idx]
    upper = minimum + error

    ax.contourf(x, y, values, levels=[minimum, upper], colors=["red"], alpha=0.3)

    if legend:
        ax.legend(
            handles=[_legend_proxy(f"$\\leq$ min + $\\sigma$ ({error:.3g})")],
            loc="upper right",
            fontsize=8,
            frameon=True,
            facecolor="white",
            edgecolor="black",
        )
    return idx, minimum, error

def add_minimum_percent_band(
    ax, x, y, values, errors, percent, *, k=1, legend=True):
    """Highlight values within a relative tolerance of the minimum."""

    values = np.asarray(values, dtype=float)
    errors = np.asarray(errors, dtype=float)

    valid = np.isfinite(values) & np.isfinite(errors)
    if not np.any(valid):
        return

    minimum = np.min(values[valid])

    idx_min = np.unravel_index(
        np.nanargmin(np.where(valid, values, np.nan)),
        values.shape
    )

    sigma_min = errors[idx_min]
    

    if minimum < 0:
        upper = minimum - (percent/100) * (minimum + k * sigma_min)
        levels = [minimum, upper]
    else:
        upper = minimum + (percent/100) * (minimum + k * sigma_min)
        levels = [minimum, upper]

    ax.contourf(
        x, y, np.ma.masked_invalid(values),
        levels=levels,
        colors=["red"],
        alpha=0.3
    )

    if legend:
        handles, labels = ax.get_legend_handles_labels()
    
        handles.append(
            _legend_proxy(
                rf"$<$ min $+ ~{percent:.1f}\%$ of min $+ {k} \sigma ~({sigma_min:.3g})$"
            )
        )
        labels.append(
                rf"$<$ min $+ ~{percent:.1f}\%$ of min $+ {k} \sigma ~({sigma_min:.3g})$"
            )
    
        ax.legend(
            handles=handles,
            labels=labels,
            loc="upper right",
            fontsize=8,
            frameon=True,
            facecolor="white",
            edgecolor="black",
        )
    return idx_min, minimum, sigma_min, upper

def add_zero_error_band(ax, x, y, values, errors, *, legend=True):
    """Highlight values statistically consistent with zero."""
    values = np.asarray(values, dtype=float)
    errors = np.asarray(errors, dtype=float)
    mask = np.abs(values) <= np.abs(errors)

    if mask.any():
        ax.contourf(x, y, mask.astype(float), levels=[0.5, 1.5],
                    colors=["red"], alpha=0.3)

    if legend:
        ax.legend(
            handles=[_legend_proxy("$|c| \\leq \\sigma$")],
            loc="upper right",
            fontsize=8,
            frameon=True,
            facecolor="white",
            edgecolor="black",
        )
    return mask

## Observables

The six panels are paired as requested: centroid x/y, sigma x/y, and
intensity next to the absolute astigmatism. The intensity is plotted with a
negated sign so that every panel can use the same "minimum is best" convention.

In [ ]:
OBSERVABLES = [
    ("centroid_x_um", "centroid_x_err_um", "centroid x", r"$x_c$ [$\mu$m]", "zero"),
    ("centroid_y_um", "centroid_y_err_um", "centroid y", r"$y_c$ [$\mu$m]", "zero"),
    ("sigma_x_um", "sigma_x_err_um", r"$\sigma_x$", r"$\sigma_x$ [$\mu$m]", "min"),
    ("sigma_y_um", "sigma_y_err_um", r"$\sigma_y$", r"$\sigma_y$ [$\mu$m]", "min"),
    ("Intensity", "Intensity_err", "Intensity", r"$-I$ [count / $\mu$m$^2$]", "neg_min"),
    ("astigmatism_mm", "astigmatism_err_mm", "Astigmatism", r"$|d|$ [mm]", "abs_min"),
]


def plot_observables(frame, title):
    fig, axs = plt.subplots(3, 2, figsize=(17, 16))
    fig.suptitle(title, fontsize=16)

    for ax, (column, err_column, name, clabel, mode) in zip(axs.flat, OBSERVABLES):
        values = to_grid(frame, column)
        errors = to_grid(frame, err_column)

        if mode == "neg_min":
            values = -values
        elif mode == "abs_min":
            values = np.abs(values)

        mesh = ax.pcolormesh(TX, RY, values, shading="auto", rasterized=True)

        if mode == "zero":
            add_zero_error_band(ax, TX, RY, values, errors)
            idx = np.unravel_index(np.argmin(np.abs(values)), values.shape)
            tx_opt, ry_opt = TX[idx], RY[idx]
            inside = bool(np.abs(values[idx]) <= errors[idx])
            stats = (
                f"closest to 0 at ({tx_opt:+.3f}, {ry_opt:+.3f}) = "
                f"{values[idx]:+.4f}  ({'inside' if inside else 'outside'} $\\sigma$)"
            )
        else:
            idx, minimum, error, upper = add_minimum_percent_band(ax, TX, RY, values, errors, percent=1)
            tx_opt, ry_opt = TX[idx], RY[idx]
            if mode == "neg_min":
                stats = (
                    f"max at ({tx_opt:+.3f}, {ry_opt:+.3f}) = "
                    f"{-values[idx]:.3f} $\\pm$ {error:.3f}"
                )
            else:
                stats = (
                    f"min at ({tx_opt:+.3f}, {ry_opt:+.3f}) = "
                    f"{minimum:.4f} $\\pm$ {error:.4f}"
                )
        if mode == 'zero':
            ax.contour(TX, RY, values, levels=10, colors="white", linewidths=0.2)
        else:
            levels = np.linspace(upper, np.nanmax(values), 15)
            ax.contour(TX, RY, values, levels=levels, colors="white", linewidths=0.2)
        ax.scatter(tx_opt, ry_opt, c="white", marker="d", s=30, zorder=5)
        ax.set_xlabel("TX [mm]")
        ax.set_ylabel("RY [mrad]")
        ax.set_title(f"{name}: {stats}")
        fig.colorbar(mesh, ax=ax, label=clabel)
        ax.set_xlim(-2, 2)
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    return fig

In [ ]:
for z_label, frame in datasets.items():
    fig = plot_observables(frame, f"Observables at {z_label}")
    fig.show()

In [ ]:
fig.savefig("/home/gabriel.amici/Documents/CEC/Observables-Focys.png", dpi=300, bbox_inches="tight")
fig.savefig("/home/gabriel.amici/Documents/CEC/Observables-Focys.pdf", dpi=300, bbox_inches="tight")
fig.savefig("/home/gabriel.amici/Documents/CEC/Observables-Focys.svg", dpi=300, bbox_inches="tight")

## Objective functions

The definitions are unchanged from `CAX_optimization.ipynb`:

$$
\begin{aligned}
f_1 &= \frac{\sigma_x \sigma_y}{I}, &
f_2 &= \frac{\sigma_x}{\sigma_x^0} + \frac{\sigma_y}{\sigma_y^0}, &
f_3 &= -\frac{I}{I_0}, \\
f_4 &= f_2 + f_3, &
f_5 &= \frac{|d|}{\Delta}, &
f_6 &= f_2 + f_5, \\
f_7 &= f_3 + f_5, &
f_8 &= f_4 + f_5.
\end{aligned}
$$

The scan stores sigma in μm; it is converted to mm internally so the formulas
and the original normalisation constants remain untouched.

In [ ]:
def objective_grids(frame):
    """Compute f1..f8 and their propagated uncertainties."""
    sx = to_grid(frame, "sigma_x_um") / 1000.0
    sy = to_grid(frame, "sigma_y_um") / 1000.0
    esx = to_grid(frame, "sigma_x_err_um") / 1000.0
    esy = to_grid(frame, "sigma_y_err_um") / 1000.0

    intensity = to_grid(frame, "Intensity")
    e_intensity = to_grid(frame, "Intensity_err")

    astig = to_grid(frame, "astigmatism_mm")
    e_astig = to_grid(frame, "astigmatism_err_mm")

    f1 = (1000.0) ** 2 * sx * sy / intensity
    f2 = 1000.0 * sx / SIGMAX_0 + 1000.0 * sy / SIGMAY_0
    f3 = -intensity / I0
    f4 = f2 + f3
    f5 = np.abs(astig) / DELTA
    f6 = f2 + f5
    f7 = f3 + f5
    f8 = f4 + f5

    # Delta-method propagation assuming independent observables.
    e1 = np.abs(f1) * np.sqrt(
        (esx / sx) ** 2 + (esy / sy) ** 2 + (e_intensity / intensity) ** 2
    )
    e2 = np.sqrt((1000.0 * esx / SIGMAX_0) ** 2 + (1000.0 * esy / SIGMAY_0) ** 2)
    e3 = e_intensity / I0
    e4 = np.sqrt(e2 ** 2 + e3 ** 2)
    e5 = e_astig / DELTA
    e6 = np.sqrt(e2 ** 2 + e5 ** 2)
    e7 = np.sqrt(e3 ** 2 + e5 ** 2)
    e8 = np.sqrt(e2 ** 2 + e3 ** 2 + e5 ** 2)

    objectives = {"f1": f1, "f2": f2, "f3": f3, "f4": f4,
                  "f5": f5, "f6": f6, "f7": f7, "f8": f8}
    errors = {"f1": e1, "f2": e2, "f3": e3, "f4": e4,
              "f5": e5, "f6": e6, "f7": e7, "f8": e8}
    return objectives, errors

In [ ]:
OBJECTIVE_LABELS = {
    "f1": r"$\sigma_x\sigma_y/I$",
    "f2": r"$\sigma_x/\sigma_x^0 + \sigma_y/\sigma_y^0$",
    "f3": r"$-I/I_0$",
    "f4": r"$f_2(\sigma_x, \sigma_y) + f_3(I)$",
    "f5": r"$|d|/\Delta$",
    "f6": r"$f_2(\sigma_x, \sigma_y) + f_5(d)$",
    "f7": r"$f_3(I) + f_5(d)$",
    "f8": r"$f_4(\sigma_x, \sigma_y, I) + f_5(d)$",
}

OBJECTIVE_NAMES = {
    "f1": "Mixed geometric intensity",
    "f2": "Geometric",
    "f3": "Intensity",
    "f4": "Geometric and intensity",
    "f5": "Astigmatism",
    "f6": "Geometric and astigmatism",
    "f7": "Intensity and astigmatism",
    "f8": "Geometric, intensity and astigmatism",
}

ORDER = [f"f{i}" for i in range(1, 9)]


def plot_objectives(frame, title):
    objectives, errors = objective_grids(frame)
    fig, axs = plt.subplots(4, 2, figsize=(17, 24))
    fig.suptitle(title, fontsize=16)

    for ax, key in zip(axs.flat, ORDER):
        values = objectives[key]
        err = errors[key]

        mesh = ax.pcolormesh(TX, RY, values, shading="auto")
        idx, minimum, uncertainty, upper = add_minimum_percent_band(
            ax, TX, RY, values, err, percent=1, k=1, legend=(key == "f1")
        )
        tx_opt, ry_opt = TX[idx], RY[idx]

        levels = np.linspace(upper, np.nanmax(values), 15)
        ax.contour(TX, RY, values, levels=levels, colors="white", linewidths=0.2)
        ax.scatter(tx_opt, ry_opt, c="white", marker="d", s=30, zorder=5)
        ax.set_xlabel("TX [mm]")
        ax.set_ylabel("RY [mrad]")
        ax.set_title(
            f"{OBJECTIVE_NAMES[key]} (${key}$): min at "
            f"({tx_opt:+.3f}, {ry_opt:+.3f}) = {minimum:.4f} $\\pm$ {uncertainty:.4f}"
        )
        fig.colorbar(mesh, ax=ax, label=OBJECTIVE_LABELS[key])
        ax.set_xlim(-2, 2)
    fig.tight_layout(rect=[0, 0, 1, 0.98])
    return fig

In [ ]:
for z_label, frame in datasets.items():
    plot_objectives(frame, f"Objectives at {z_label}")
    plt.show()

In [ ]:
plt.rcParams.update({'text.usetex': True, 'font.family': 'serif', 'font.serif': ['Computer Modern'], 'font.size': 12})

In [ ]:
OBSERVABLE_QUAD = [
    ("sigma_x_um", "sigma_x_err_um", r"$\sigma_x$", r"$\sigma_x$ [$\mu$m]", "min"),
    ("sigma_y_um", "sigma_y_err_um", r"$\sigma_y$", r"$\sigma_y$ [$\mu$m]", "min"),
    ("Intensity", "Intensity_err", "Intensity", r"$-I$ [count / $\mu$m$^2$]", "neg_min"),
    ("Objective", "objective_err", "Objective function", r"$f_1(\sigma_x, \sigma_y, I)=\sigma_x\sigma_y/I$", "min"),
]

# Just the data corresponding to OBSERVABLE_QUAD
frame = df_zf[["TX_mm", "RY_mrad"] + [col for col, _, _, _, _ in OBSERVABLE_QUAD[:-1]] + [err_col for _, err_col, _, _, _ in OBSERVABLE_QUAD[:-1]]].copy()

# Append the objective data and error
frame["Objective"] = objective_grids(df_zf)[0]["f1"].flatten()
frame["objective_err"] = objective_grids(df_zf)[1]["f1"].flatten()

# fig, axs = plt.subplots(1, 4, figsize=(20, 6), sharey=True, layout="constrained")
fig, axs = plt.subplots(1, 4, figsize=(14, 4), sharey=True, layout="constrained")
# fig.suptitle(title, fontsize=16)

for ax, (column, err_column, name, clabel, mode) in zip(axs.flat, OBSERVABLE_QUAD):
    values = to_grid(frame, column)
    errors = to_grid(frame, err_column)

    if mode == "neg_min":
        values = -values
    elif mode == "abs_min":
        values = np.abs(values)

    mesh = ax.pcolormesh(TX, RY, values, shading="auto", rasterized=True)

    if mode == "zero":
        add_zero_error_band(ax, TX, RY, values, errors)
        idx = np.unravel_index(np.argmin(np.abs(values)), values.shape)
        tx_opt, ry_opt = TX[idx], RY[idx]
        inside = bool(np.abs(values[idx]) <= errors[idx])
        stats = (
            f"closest to 0 at ({tx_opt:+.3f}, {ry_opt:+.3f}) = "
            f"{values[idx]:+.4f}  ({'inside' if inside else 'outside'} $\\sigma$)"
        )
    else:
        idx, minimum, error, upper = add_minimum_percent_band(ax, TX, RY, values, errors, percent=5)
        tx_opt, ry_opt = TX[idx], RY[idx]
        if mode == "neg_min":
            stats = (
                f"max at ({tx_opt:+.2f}, {ry_opt:+.2f}) = "
                f"{-values[idx]:.2f} $\\pm$ {error:.2f}"
            )
        else:
            stats = (
                f"min at ({tx_opt:+.2f}, {ry_opt:+.2f}) = "
                f"{minimum:.2f} $\\pm$ {error:.2f}"
            )
    if mode == 'zero':
        ax.contour(TX, RY, values, levels=10, colors="white", linewidths=0.2)
    else:
        levels = np.linspace(upper, np.nanmax(values), 15)
        ax.contour(TX, RY, values, levels=levels, colors="white", linewidths=0.2)
    ax.scatter(tx_opt, ry_opt, c="white", marker="d", s=30, zorder=5)
    ax.set_xlabel("TX [mm]")
    # Give every panel its own horizontal scale without taking plot width.
    fig.colorbar(
        mesh, ax=ax, label=clabel, orientation="horizontal",
        fraction=0.05, pad=0.05, aspect=30,
    )
    # ax.set_title(f"{name}: {stats}")
    ax.set_title(f"{name}")
axs[0].set_ylabel("RY [mrad]")

In [ ]:
fig.savefig("/home/gabriel.amici/Documents/CEC/Observable-panel.pdf", dpi=300, bbox_inches="tight")

## Minima summary

In [ ]:
def observable_minima(frame):
    rows = []
    for column, err_column, name, _, mode in OBSERVABLES:
        values = to_grid(frame, column)
        errors = to_grid(frame, err_column)

        if mode == "neg_min":
            values = -values
        elif mode == "abs_min":
            values = np.abs(values)

        if mode == "zero":
            idx = np.unravel_index(np.nanargmin(np.abs(values)), values.shape)
            criterion = "closest to 0"
            value = values[idx]
            error = errors[idx]
        else:
            idx = np.unravel_index(np.nanargmin(values), values.shape)
            criterion = "maximum" if mode == "neg_min" else "minimum"
            value = -values[idx] if mode == "neg_min" else values[idx]
            error = errors[idx]

        rows.append({
            "observable": name,
            "criterion": criterion,
            "TX [mm]": TX[idx],
            "RY [mrad]": RY[idx],
            "value": value,
            "error": error,
        })
    return pd.DataFrame(rows)


def objective_minima(frame):
    objectives, errors = objective_grids(frame)
    rows = []
    for key in ORDER:
        values = objectives[key]
        idx = np.unravel_index(np.nanargmin(values), values.shape)
        rows.append({
            "objective": key,
            "TX [mm]": TX[idx],
            "RY [mrad]": RY[idx],
            "minimum": values[idx],
            "propagated error": errors[key][idx],
        })
    return pd.DataFrame(rows)

In [ ]:
for z_label, frame in datasets.items():
    print(f"\n{'=' * 70}\n{z_label}\n{'=' * 70}")
    print("\nObservable optima")
    display(observable_minima(frame).round(4))
    print("\nObjective minima")
    display(objective_minima(frame).round(4))

In [ ]:
comparison = pd.concat(
    {
        z_label: objective_minima(frame).set_index("objective")
        for z_label, frame in datasets.items()
    },
    axis=1,
)
display(comparison.round(4))

In [ ]:
print("Objective minima comparison\n" + "=" * 60)
for key in ["f1", "f4", "f8"]:
    print(f"\n{key}")
    for z_label, frame in datasets.items():
        objectives, errors = objective_grids(frame)
        values = objectives[key]
        idx = np.unravel_index(np.nanargmin(values), values.shape)
        print(
            f"  {z_label:>12s}: ({TX[idx]:+.3f} mm, {RY[idx]:+.3f} mrad) = "
            f"{values[idx]:.4f} +/- {errors[key][idx]:.4f}"
        )

In [ ]:
# Single objective plot at selected z position

def plot_single_objective(frame, objective_key, title, fig=None, ax=None):
    objectives, errors = objective_grids(frame)
    values = objectives[objective_key]
    err = errors[objective_key]

    if fig is None or ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))
    mesh = ax.pcolormesh(TX, RY, values, shading="auto")
    idx, minimum, uncertainty, upper = add_minimum_percent_band(
        ax, TX, RY, values, err, percent=5, k=1
    )
    tx_opt, ry_opt = TX[idx], RY[idx]

    levels = np.linspace(upper, np.nanmax(values), 15)
    ax.contour(TX, RY, values, levels=levels, colors="white", linewidths=0.2)
    ax.scatter(tx_opt, ry_opt, c="white", marker="d", s=30, zorder=5)
    ax.set_xlabel("TX [mm]")
    ax.set_ylabel("RY [mrad]")
    ax.set_title(
        f"{OBJECTIVE_NAMES[objective_key]} (${objective_key}$): min at "
        f"({tx_opt:+.3f}, {ry_opt:+.3f}) = {minimum:.4f} $\\pm$ {uncertainty:.4f}"
    )
    fig.colorbar(mesh, ax=ax, label=OBJECTIVE_LABELS[objective_key])
    ax.set_xlim(-2, 2)
    fig.tight_layout()
    return fig, ax

In [ ]:
%matplotlib qt5
# %matplotlib inline

In [ ]:
fig, ax = plot_single_objective(datasets[f"z = {Z_FOCUS:g} mm"], "f1", f"Single objective f1 at z = {Z_FOCUS:g} mm")

In [ ]:
fig.savefig("/home/gabriel.amici/Documents/CEC/Objective.pdf", dpi=300, bbox_inches="tight")
fig.savefig("/home/gabriel.amici/Documents/CEC/Objective.png", dpi=300, bbox_inches="tight")
fig.savefig("/home/gabriel.amici/Documents/CEC/Objective.svg", dpi=300, bbox_inches="tight")


In [ ]:
f1_min_idx = np.argmin(objective_grids(datasets[f"z = {Z_FOCUS:g} mm"])[0]["f1"])

f1_min_pos = (np.ravel(TX)[f1_min_idx], np.ravel(RY)[f1_min_idx])
f1_min_pos

In [ ]:
from shadow_interface.si_beamlines import CAXSim

In [ ]:
cax = CAXSim(total_rays=100_000, image_size=(210, 280))

tx, ry = [-0.39813492, -0.01101705]

cax.mirror.tx = 0#tx#-1.0765 #f1_min_pos[0]
cax.mirror.ry = 25.255e-3#ry#-0.0265 #f1_min_pos[1]

cax.trace()

ana = fit_beam_image(cax.dvf_B1.image)

In [ ]:
ana.fit(ana.compute_momenta())
ana.plot(ana.hprm_fitting)
ana.print_stats(ana.hprm_fitting)

In [ ]:
1.1128e+01*1.8768e+01/4.8735e+01

# Optimization

Each optimizer keeps measurement histories in `observables` and
`observable_errors`, dictionaries of lists keyed by `f1`, `peak`,
`sig_major` and `sig_minor`. Widths are in µm and peak is in counts/µm²;
errors are absolute uncertainties in the same units as their values.
Each call to `objective_function` appends one sample, including `NaN` values
when the beam is not visible. RCDS still receives only the scalar `f1`.

After running, use `opt.observables["sig_major"]` and
`opt.observable_errors["sig_major"]` for a width series and its error bars,
or `pd.DataFrame(opt.observables)` for all series indexed by measurement.
The prealigned and 5D optimizers use the same recording logic.


In [ ]:
# Objective and monitored observables at the DVF position.
def f_1(cax: CAXSim, z_pos: float):
    """Return one sample of observables and their absolute uncertainties."""
    observables = dict.fromkeys(("f1", "peak", "sig_major", "sig_minor"), np.nan)
    observable_errors = observables.copy()

    beamr = cax.dvf_B1.beam.duplicate()
    beamr.retrace(z_pos)
    image = image_from_beam(beamr, cax.dvf_B1.frame, image_size=(210, 280))
    ana = fit_beam_image(image)

    if not ana.beam_visible or ana.hprm_fitting is None:
        return observables, observable_errors

    fit = ana.hprm_fitting
    errors = fit.get("errors", {})
    for name, key in (("peak", "amplitude"),
                      ("sig_major", "sig_major"),
                      ("sig_minor", "sig_minor")):
        observables[name] = fit[key]
        observable_errors[name] = errors.get(key, np.nan)

    peak = observables["peak"]
    sig_major = observables["sig_major"]
    sig_minor = observables["sig_minor"]
    if not all(np.isfinite(value) and value > 0
               for value in (peak, sig_major, sig_minor)):
        return observables, observable_errors

    f1 = sig_major * sig_minor / peak
    observables["f1"] = f1
    # Delta-method propagation, assuming independent observables.
    observable_errors["f1"] = abs(f1) * np.sqrt(
        (observable_errors["peak"] / peak) ** 2
        + (observable_errors["sig_major"] / sig_major) ** 2
        + (observable_errors["sig_minor"] / sig_minor) ** 2
    )
    return observables, observable_errors


In [ ]:
f1_err = np.nanmean(objective_grids(datasets[f"z = {Z_FOCUS:g} mm"])[1]['f1'])

print(f"f1 error: {f1_err:.4g}")

In [ ]:
params = RCDSParams()

# motor limits: TX and RY
params.limit_lower = [-2, -0.1] # -4 mm and -0.1 mrad and -15cm
params.limit_upper = [2, 0.1] # 4 mm and 0.1 mrad and 15cm
params.caxsim = CAXSim(total_rays=100_000, image_size=(210, 280))
params.z_pos = Z_FOCUS
# RCDS params
# these are the parameters RCDS requires for its internal
# logic and operations
params.max_number_evals = 500
params.max_number_iters = 13
params.initial_stepsize = 0.5
params.noise_level = 0.03
params.tolerance = 1e-8
params.update_search_directions = True
params.update_search_directions = True

# initial position refers to the initial point in the
# optimaztion variables space
# you could set it to a particular value or sample
# the available search space ramdomly, as below
# params.initial_position = np.random.uniform(
#     low=params.limit_lower, high=params.limit_upper)
params.initial_position = np.random.uniform(
    low=params.limit_lower, high=params.limit_upper)
# params.initial_position = np.array([1.5, -0.075])
# search directions along whith to start the line
# minimizations. It is a matrix with each row standing for a
# given direction

# initial directions are purely TX and RY
params.initial_search_directions = np.eye(2)
# params.initial_search_directions = np.array([dir2, dir1])

print(params)

In [ ]:
class CAXRCDS(RCDS):

    def __init__(self, use_thread=True, isonline=True):
        super().__init__(use_thread=use_thread, isonline=isonline)
        self.params = params
        self.observables = {}
        self.observable_errors = {}

    def objective_function(self, pos):
        tx, ry = pos
        caxsim = self.params.caxsim
        caxsim.mirror.tx = tx
        caxsim.mirror.ry = ry
        caxsim.trace()
        return self._measure_objective()

    def _measure_objective(self):
        """Append one sample per measurement and return the scalar objective."""
        values, errors = f_1(self.params.caxsim, self.params.z_pos)
        for name, value in values.items():
            self.observables.setdefault(name, []).append(value)
            self.observable_errors.setdefault(name, []).append(errors[name])
        self.params.noise_level = errors['f1']
        return values["f1"]


In [ ]:
opt = CAXRCDS(isonline=False, use_thread=False)

opt.start()

In [ ]:
opt.plot_history()
plt.show()

In [ ]:
print("Optimization Overview \n" + "="*50)
f1_grid = objective_grids(datasets["z = -142 mm"])[0]['f1']
print(f"{'Evaluations':<20}: {opt.num_objective_evals}")
print()
print(f"{'Initial objective':<20}: {opt.objfuncs_evaluated[0]:.4f}")
print(f"{'Best objective':<20}: {opt.objfuncs_best[-1]:.4f}")
print(f"{'f1 array minimum':<20}: {np.nanmin(f1_grid):.4f}")
print()
print(f"{'Initial position':<20}: {opt.positions_evaluated[0]}")
print(f"{'Best position':<20}: {opt.positions_best[-1]}")
# argmin = np.unravel_index(np.argmin(f1_dvf), f1_dvf.shape)
# print(f"{'f1 position':<20}: [{tx_vals[argmin]:.8f} {ry_vals[argmin]:.8f}]")



In [ ]:
plot_optimization_data(opt, 5)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patheffects
import numpy as np

In [ ]:
def plot_knobspace_slice(optimizer, dir_idcs=(1, 2), obj_idx=None):
    """Plot a 2D slice of the optimizer's parameter space."""

    idx1, idx2 = np.array(dir_idcs) - 1

    pos_eval = np.asarray(optimizer.positions_evaluated)
    pos_best = np.asarray(optimizer.positions_best)
    objfuncs_eval = np.asarray(optimizer.objfuncs_evaluated)

    if obj_idx is not None:
        objfuncs_eval = objfuncs_eval[:, obj_idx]

    _, pos_cum_opt, _ = optimizer.get_cumul_optima(obj_idx)

    # Coordinates
    x_eval, y_eval = pos_eval[:, idx1], pos_eval[:, idx2]
    x_best, y_best = pos_best[:, idx1], pos_best[:, idx2]
    x_cum, y_cum = pos_cum_opt[:, idx1], pos_cum_opt[:, idx2]

    fig, ax = plt.subplots()

    # Evaluated positions
    ax.scatter(
        x_eval, y_eval,
        c=objfuncs_eval,
        cmap="viridis",
        label="positions evaluated",
        zorder=1
    )

    # Iteration-end optima
    ax.plot(
        x_best, y_best,
        "x", color="red",
        label="end-of-iteration optima",
        zorder=3
    )

    # Number iteration-end optima
    for i, (x, y) in enumerate(zip(x_best, y_best)):
        if i%2 == 0:  # Only annotate every other point to avoid clutter
            ax.annotate(
                str(i),
                (x, y),
                xytext=(4, 4),
                textcoords="offset points",
                fontsize=8,
                color="white",
                path_effects=[
                    plt.matplotlib.patheffects.withStroke(
                        linewidth=1.5, foreground="black"
                    )
                ],
                zorder=5
            )

    # Connect consecutive iteration-end optima
    for i in range(len(x_best) - 1):
        ax.annotate(
            "",
            xy=(x_best[i + 1], y_best[i + 1]),
            xytext=(x_best[i], y_best[i]),
            arrowprops={
                "arrowstyle": "->",
                "color": "white",
                "lw": 0.8,
                "shrinkA": 6,
                "shrinkB": 6
            },
            zorder=4
        )

    # Cumulative optima
    ax.plot(
        x_cum, y_cum,
        "o", color="red", mfc="none",
        label="cumulative optima",
        zorder=4
    )

    # Final optimum
    ax.plot(
        x_best[-1], y_best[-1],
        "d", color="magenta",
        markersize=10,
        label="final optimum",
        zorder=6
    )

    ax.set_xlabel(f"dir {idx1 + 1}")
    ax.set_ylabel(f"dir {idx2 + 1}")
    ax.set_title("parameter space slice")

    return fig, ax

In [ ]:
fig, ax = plot_knobspace_slice(optimizer=opt, dir_idcs=(1, 2))
fig, ax = plot_single_objective(datasets[f"z = {Z_FOCUS:g} mm"], "f1", f"Single objective f1 at z = {Z_FOCUS:g} mm", fig, ax)
fig.set_size_inches(12, 8)

# handles, labels = ax.get_legend_handles_labels()
# ax.set_xlim(-1.5, -0.5)
# ax.set_ylim(-0.1, 0.03)
# ax.legend(handles, labels, loc="best")


In [ ]:
fig.savefig("/home/gabriel.amici/Documents/CEC/2D-Optimization.pdf", dpi=300, bbox_inches="tight")
fig.savefig("/home/gabriel.amici/Documents/CEC/2D-Optimization.png", dpi=300, bbox_inches="tight")
fig.savefig("/home/gabriel.amici/Documents/CEC/2D-Optimization.svg", dpi=300, bbox_inches="tight")


In [ ]:
opt.num_iterations

# TY - RX Optimization

In the real beamline we have other means of pre-aliging the TX and RY motors.

Let's see what the optimizer does when we let it mess with the other controls
(primarily TY and RX, which couple together to affect the beam) while keeping
TX and RY fixed near an optimal condition.

In [ ]:
cax = CAXSim(total_rays=100_000, image_size=(210, 280))

cax.mirror.tx = 0.0
cax.mirror.ry = 25.255e-3 # 25 μrad

cax.trace()

ana = fit_beam_image(cax.dvf_B1.image)

In [ ]:
ana.fit(ana.compute_momenta())
ana.plot(ana.hprm_fitting)
ana.print_stats(ana.hprm_fitting)

In [ ]:
cax.mirror.tilt

In [ ]:
[cax.mirror.shadow_oe.X_ROT, cax.mirror.shadow_oe.Y_ROT, cax.mirror.shadow_oe.Z_ROT]

In [ ]:
cax.mirror.ty = 0.5
cax.mirror.rx = -0.1
cax.mirror.rz = 0.1
cax.trace()
ana = fit_beam_image(cax.dvf_B1.image)
ana.fit(ana.compute_momenta())
ana.plot(ana.hprm_fitting)
ana.print_stats(ana.hprm_fitting)

In [ ]:
params_prealigned = RCDSParams()

# motor limits: TY and RX
params_prealigned.limit_lower = [-0.5, -0.1] # -0.5 mm and -0.1 mrad
params_prealigned.limit_upper = [0.5, 0.1]   # +0.5 mm and +0.1 mrad
params_prealigned.caxsim = CAXSim(total_rays=100_000, image_size=(210, 280))
params_prealigned.caxsim.mirror.ry = 25.255e-3 # 25.25 μrad
params_prealigned.z_pos = Z_FOCUS
# RCDS params
# these are the parameters RCDS requires for its internal
# logic and operations
params_prealigned.max_number_evals = 500
params_prealigned.max_number_iters = 13
params_prealigned.initial_stepsize = 0.5
params_prealigned.noise_level = 0.001
params_prealigned.tolerance = 1e-8
params_prealigned.update_search_directions = True
params_prealigned.update_search_directions = True

# initial position refers to the initial point in the
# optimaztion variables space
# you could set it to a particular value or sample
# the available search space ramdomly, as below
# params.initial_position = np.random.uniform(
#     low=params.limit_lower, high=params.limit_upper)
params_prealigned.initial_position = np.random.uniform(
    low=params_prealigned.limit_lower, high=params_prealigned.limit_upper)
# params.initial_position = np.array([1.5, -0.075])
# search directions along whith to start the line
# minimizations. It is a matrix with each row standing for a
# given direction

# initial directions are purely TX and RY
params_prealigned.initial_search_directions = np.eye(2)
# params.initial_search_directions = np.array([dir2, dir1])

print(params_prealigned)

In [ ]:
class CAXRCDS_prealigned(CAXRCDS):

    def __init__(self, use_thread=True, isonline=True):
        super().__init__(use_thread=use_thread, isonline=isonline)
        self.params = params_prealigned

    def objective_function(self, pos):

        ty, rx = pos
        caxsim = self.params.caxsim

        caxsim.mirror.ty = ty
        caxsim.mirror.rx = rx

        caxsim.trace()

        return self._measure_objective()

In [ ]:
opt = CAXRCDS_prealigned(use_thread=False, isonline=False)

opt.start()

In [ ]:
print("Optimization Overview \n" + "="*50)
f1_grid = objective_grids(datasets["z = -142 mm"])[0]['f1']
print(f"{'Evaluations':<20}: {opt.num_objective_evals}")
print()
print(f"{'Initial objective':<20}: {opt.objfuncs_evaluated[0]:.4f}")
print(f"{'Best objective':<20}: {opt.objfuncs_best[-1]:.4f}")
print(f"{'f1 array minimum':<20}: {np.nanmin(f1_grid):.4f}")
print()
print(f"{'Initial position':<20}: {opt.positions_evaluated[0]}")
print(f"{'Best position':<20}: {opt.positions_best[-1]}")
# argmin = np.unravel_index(np.argmin(f1_dvf), f1_dvf.shape)
# print(f"{'f1 position':<20}: [{tx_vals[argmin]:.8f} {ry_vals[argmin]:.8f}]")


In [ ]:
opt.plot_history()
plt.show()

In [ ]:
plot_optimization_data(opt, 3)

In [ ]:
# fig, ax = plot_knobspace_slice(optimizer=opt, dir_idcs=(1, 2))
# fig, ax = plot_single_objective(datasets[f"z = {Z_FOCUS:g} mm"], "f1", f"Single objective f1 at z = {Z_FOCUS:g} mm", fig, ax)
# fig.set_size_inches(12, 8)

# handles, labels = ax.get_legend_handles_labels()
# ax.set_xlim(-1.5, -0.5)
# ax.set_ylim(-0.1, 0.03)
# ax.legend(handles, labels, loc="best")


In [ ]:
cax = CAXSim(total_rays=100_000, image_size=(210, 280))

cax.mirror.tx = 0.0
cax.mirror.ry = 25.25e-3 # 25 μrad

cax.trace()

ana = fit_beam_image(cax.dvf_B1.image)

<!-- # 5D optimization -->

In [ ]:
ana.fit(ana.compute_momenta())
ana.plot(ana.hprm_fitting)
ana.print_stats(ana.hprm_fitting)

In [ ]:
opt.objfuncs_best[-1]

In [ ]:
opt_pos = opt.positions_best[-1]
ty, rx = opt_pos

cax.mirror.ty = ty
cax.mirror.rx = rx

cax.trace()
ana = fit_beam_image(cax.dvf_B1.image)
ana.fit(ana.compute_momenta())
ana.plot(ana.hprm_fitting)
ana.print_stats(ana.hprm_fitting)

In [ ]:
cax.mirror.offset, cax.mirror.tilt

In [ ]:
params5D = RCDSParams()

# motor limits: TX, TY, RX, RY, RZ
params5D.limit_lower = [-2, -1, -0.5, -0.1, -0.5] # -4 mm and -0.1 mrad and -15cm
params5D.limit_upper = -params5D.limit_lower # 4 mm and 0.1 mrad and 15cm
params5D.caxsim = CAXSim(total_rays=100_000, image_size=(210, 280))
params5D.z_pos = Z_FOCUS
# RCDS params
# these are the parameters RCDS requires for its internal
# logic and operations
params5D.max_number_evals = 600
params5D.max_number_iters = 20
params5D.initial_stepsize = 0.5
params5D.noise_level = 0.001
params5D.tolerance = 1e-8
params5D.update_search_directions = True
params5D.update_search_directions = True

# initial position refers to the initial point in the
# optimaztion variables space
# you could set it to a particular value or sample
# the available search space ramdomly, as below
# params.initial_position = np.random.uniform(
#     low=params.limit_lower, high=params.limit_upper)
params5D.initial_position = np.random.uniform(
    low=params5D.limit_lower, high=params5D.limit_upper)

# params5D.initial_position = np.array([-1.9, 0.0, 0.0, 0.09, 0.0])
# params5D.initial_position = np.array([-3.6, 0.09])
# search directions along whith to start the line
# minimizations. It is a matrix with each row standing for a
# given direction

# initial directions are purely TX and RY
params5D.initial_search_directions = np.eye(5)
# params5D.initial_search_directions = np.array([dir2, dir1])

print(params5D)

In [ ]:
class CAXRCDS5D(CAXRCDS):

    def __init__(self, use_thread=True, isonline=True):
        super().__init__(use_thread=use_thread, isonline=isonline)
        self.params = params5D

    def objective_function(self, pos):

        tx, ty, rx, ry, rz = pos
        caxsim = self.params.caxsim

        caxsim.mirror.tx = tx
        caxsim.mirror.ty = ty
        caxsim.mirror.rx = rx
        caxsim.mirror.ry = ry
        caxsim.mirror.rz = rz

        caxsim.trace()

        return self._measure_objective()

In [ ]:
opt5D = CAXRCDS5D(isonline=False, use_thread=False)
opt5D.start()
# opt5D.start()

In [ ]:
opt5D.num_objective_evals

In [ ]:
print("Optimization Overview \n" + "="*50)
f1_grid = objective_grids(datasets["z = -142 mm"])[0]['f1']
print(f"{'Evaluations':<20}: {opt5D.num_objective_evals}")
print()
print(f"{'Initial objective':<20}: {opt5D.objfuncs_evaluated[0]:.4f}")
print(f"{'Best objective':<20}: {opt5D.objfuncs_best[-1]:.4f}")
print(f"{'f1 array minimum':<20}: {np.nanmin(f1_grid):.4f}")
print()
print(f"{'Initial position':<20}: {opt5D.positions_evaluated[0]}")
print(f"{'Best position':<20}: {opt5D.positions_best[-1]}")
# argmin = np.unravel_index(np.argmin(f1_dvf), f1_dvf.shape)
# print(f"{'f1 position':<20}: [{tx_vals[argmin]:.8f} {ry_vals[argmin]:.8f}]")


In [ ]:
opt5D.positions_evaluated

In [ ]:
plt.imshow(params5D.caxsim.dvf_B1.image.histogram, cmap='gray')

In [ ]:
params5D.caxsim.mirror.tx, params5D.caxsim.mirror.ty, params5D.caxsim.mirror.rx, params5D.caxsim.mirror.ry, params5D.caxsim.mirror.rz

In [ ]:
opt5D.plot_history()
plt.show()

In [ ]:
fig, ax = plot_knobspace_slice(optimizer=opt5D, dir_idcs=(1, 4))
fig.set_size_inches(12, 8)
fig, ax = plot_single_objective(datasets[f"z = {Z_FOCUS:g} mm"], "f1", f"Single objective f1 at z = {Z_FOCUS:g} mm", fig, ax)

# handles, labels = ax.get_legend_handles_labels()

# ax.legend(handles, labels, loc="best")


In [ ]:
opt5D.positions_best[-1], opt5D.objfuncs_best[-1]

In [ ]:
cax = CAXSim(total_rays=100_000, image_size=(210, 280))

ana0 = fit_beam_image(cax.dvf_B1.image)

tx, ty, rx, ry, rz = opt5D.positions_best[-1]

cax.mirror.offset = [tx, ty, 0.]
cax.mirror.tilt = [rx, ry, rz]

cax.trace()

anaf = fit_beam_image(cax.dvf_B1.image)

In [ ]:
print(f"Initial position: {opt5D.positions_evaluated[0]}")

ana0.plot(ana0.hprm_fitting)
ana0.print_stats(ana0.hprm_fitting)

print('='*70 + f"\nBest position: {opt5D.positions_best[-1]}\n" + '='*70)


anaf.plot(anaf.hprm_fitting)

anaf.print_stats(anaf.hprm_fitting)

In [ ]:
def plot_optimization_data(opt, k):

    fig, axs = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
    fig.subplots_adjust(wspace=0.3, hspace=0.1)
    
    x = np.arange(len(opt.observables['f1']))[::k]

    sigx = np.array(opt.observables['sig_major'])[::k]
    sigy = np.array(opt.observables['sig_minor'])[::k]

    sigx_err = np.array(opt.observable_errors['sig_major'])[::k]
    sigy_err = np.array(opt.observable_errors['sig_minor'])[::k]

    ax00r = axs[0, 0].twinx()

    axs[0, 0].errorbar(x, sigx, yerr=sigx_err, marker='o', c='blue')
    ax00r.errorbar(x, sigy, yerr=sigy_err, marker='s', c='red')

    axs[0, 0].set_ylabel(r"$\sigma_x ~[\mu m]$", c='blue')
    ax00r.set_ylabel(r"$\sigma_y ~[\mu m]$", c='red')

    axs[0, 0].tick_params(axis="y", colors='blue')
    ax00r.tick_params(axis="y", colors='red')

    axs[0, 0].set_xlim(x[0], x[-1])
    # axs[0, 0].set_xlim(100, x[-1])

    # axs[0, 0].set_ylim(18.2, 18.7)
    # ax00r.set_ylim(10.6, 11.1)

    axs[0, 0].grid(which='both', color='blue')
    ax00r.grid(which='both', color='red')

    peak = np.array(opt.observables['peak'])[::k]
    peak_err = np.array(opt.observables['peak'])[::k]

    axs[0, 1].errorbar(x, peak, yerr=peak_err/peak, marker='o')
    # axs[0, 1].set_ylim(48, 53)
    axs[0, 1].grid(which='both')
    axs[0, 1].set_ylabel(r"$I ~[\textrm{count}/\mu m ^2]$")


    sig_xy = sigx*sigy
    sig_xy_err = np.sqrt((sigx_err/sigx)**2 + (sigy_err/sigy)**2)

    axs[1, 0].errorbar(x, sig_xy, yerr=sig_xy_err*sig_xy, marker='o')
    # axs[1, 0].set_ylim(195, 205)
    axs[1, 0].grid(which='both')
    axs[1, 0].set_ylabel(r"$\sigma_x\sigma_y ~[\mu m ^2]$")

    f1      = np.array(opt.observables['f1'])[::k]
    f1_err  = np.array(opt.observable_errors['f1'])[::k]

    axs[1, 1].errorbar(x, f1, yerr=f1_err, marker='o')
    # axs[1, 1].set_ylim(3.7, 4.2)
    axs[1, 1].grid(which='both')
    axs[1, 1].set_ylabel(r"$f_1(\sigma_x, \sigma_y, I) ~[\mu m ^4/\textrm{count}]$")

    axs[1, 0].set_xlabel('Iteration')
    axs[1, 1].set_xlabel('Iteration')

In [ ]:
opt = opt5D

